# Agile Testing Using Machine Learning

## Software Defect Prediction with Data Leakage Prevention and Hyperparameter Tuning

In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

import joblib
import matplotlib.pyplot as plt


In [ ]:

# Load Dataset (Upload kc1.csv in Colab or same folder)
data = pd.read_csv('kc1.csv')

print("Dataset Shape:", data.shape)
data.head()


In [ ]:

# Dataset Information
data.info()


In [ ]:

# Check Missing Values
print("Total Missing Values:", data.isnull().sum().sum())


In [ ]:

# Separate Features and Target
X = data.drop('defects', axis=1)
y = data['defects']

print("Target Distribution:\n", y.value_counts())


In [ ]:

# Train-Test Split (Prevents Data Leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)


In [ ]:

# Handle Class Imbalance using SMOTE (Only on Training Set)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("After SMOTE Training Target Distribution:\n", pd.Series(y_train_res).value_counts())


In [ ]:

# Feature Scaling (Fit only on Training Data to prevent leakage)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)


In [ ]:

# Hyperparameter Tuning using GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train_res)

print("Best Parameters:", grid_search.best_params_)

best_model = grid_search.best_estimator_


In [ ]:

# Evaluate Tuned Model on Test Set
y_pred = best_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


In [ ]:

# Save Final Trained Model
joblib.dump(best_model, "Agile_Testing_ML_Model.pkl")
print("Model saved successfully!")


In [ ]:

# Feature Importance Visualization
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10,6))
plt.title("Feature Importance in Defect Prediction")
plt.bar(range(X.shape[1]), importances[indices])
plt.xlabel("Software Metrics Features")
plt.ylabel("Importance Score")
plt.show()
